# Ecuaci?n de calor 1D: soluci?n completa

Implementaci?n docente del esquema FTCS, con an?lisis de error y animaci?n.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
def exact_solution(x, t, alpha=1.0):
    return np.exp(-alpha*np.pi**2*t) * np.sin(np.pi*x)

def initial_condition(x):
    return np.sin(np.pi*x)

def build_grids(L, T, N, dt):
    dx = L / (N + 1)
    x = np.linspace(dx, L - dx, N)
    nt = max(1, int(np.ceil(T / dt)))
    return x, nt

def ftcs_step(u, r):
    unew = np.empty_like(u)
    unew[0] = u[0] + r * (u[1] - 2.0*u[0] + 0.0)
    unew[-1] = u[-1] + r * (0.0 - 2.0*u[-1] + u[-2])
    unew[1:-1] = u[1:-1] + r * (u[2:] - 2.0*u[1:-1] + u[:-2])
    return unew

def solve_heat_ftcs(alpha=1.0, L=1.0, T=0.12, N=80, dt=None, store_every=6):
    if dt is None:
        dx_local = L / (N + 1)
        dt = 0.45 * dx_local**2 / alpha

    x, nt = build_grids(L, T, N, dt)
    dx_local = L / (N + 1)
    r_local = alpha * dt / dx_local**2

    if r_local > 0.5:
        raise ValueError(f"Esquema inestable: r={r_local:.4f} > 0.5")

    u = initial_condition(x)
    U_hist = [u.copy()]
    t_hist = [0.0]

    for n in range(1, nt + 1):
        u = ftcs_step(u, r_local)
        if n % store_every == 0 or n == nt:
            U_hist.append(u.copy())
            t_hist.append(n * dt)

    t_final = nt * dt
    return x, u, t_final, r_local, np.array(t_hist), np.array(U_hist)

In [ ]:
alpha = 1.0
L = 1.0
T = 0.12
N = 80

dx = L / (N + 1)
dt = 0.45 * dx**2 / alpha

x, u_num, t_final, r, t_hist, U_hist = solve_heat_ftcs(alpha=alpha, L=L, T=T, N=N, dt=dt)
u_ex = exact_solution(x, t_final, alpha=alpha)

err_l2 = np.sqrt(np.mean((u_num - u_ex)**2))
err_inf = np.max(np.abs(u_num - u_ex))

print(f"dx={dx:.6f}, dt={dt:.8f}, r={r:.4f}")
print(f"t_final={t_final:.6f}")
print(f"Error L2   = {err_l2:.3e}")
print(f"Error Linf = {err_inf:.3e}")

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(x, u_num, 'o-', ms=3, label='Numerica FTCS')
plt.plot(x, u_ex, '--', lw=2, label='Exacta')
plt.xlabel('x')
plt.ylabel('u(x,T)')
plt.title('Comparacion numerica vs exacta')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

In [ ]:
# Estudio de error con refinamiento de malla (r fijo)
Ns = [20, 40, 80, 160]
errs = []
hs = []

for Nloc in Ns:
    dxloc = L / (Nloc + 1)
    dtloc = 0.45 * dxloc**2 / alpha
    xloc, uloc, tloc, rloc, _, _ = solve_heat_ftcs(alpha=alpha, L=L, T=T, N=Nloc, dt=dtloc)
    uexloc = exact_solution(xloc, tloc, alpha=alpha)
    errloc = np.sqrt(np.mean((uloc - uexloc)**2))
    errs.append(errloc)
    hs.append(dxloc)

plt.figure(figsize=(6,4))
plt.loglog(hs, errs, 'o-', label='Error L2')
plt.loglog(hs, np.array(hs)**2 * errs[0] / (hs[0]**2), '--', label='Referencia O(h^2)')
plt.xlabel('h')
plt.ylabel('Error')
plt.title('Convergencia espacial (r fijo)')
plt.grid(alpha=0.25, which='both')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
line_num, = ax.plot([], [], lw=2, label='FTCS')
line_ex, = ax.plot([], [], '--', lw=1.8, label='Exacta')
text_t = ax.text(0.02, 0.92, '', transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.set_title('Evolucion temporal de temperatura')
ax.grid(alpha=0.25)
ax.legend()

def init():
    line_num.set_data([], [])
    line_ex.set_data([], [])
    text_t.set_text('')
    return line_num, line_ex, text_t

def update(i):
    ti = t_hist[i]
    line_num.set_data(x, U_hist[i])
    line_ex.set_data(x, exact_solution(x, ti, alpha=alpha))
    text_t.set_text(f't={ti:.4f}')
    return line_num, line_ex, text_t

ani = FuncAnimation(fig, update, frames=len(t_hist), init_func=init, interval=80, blit=True)
plt.close(fig)
ani

## Cierre did?ctico

- El esquema FTCS es simple y muy ?til para introducir difusi?n.
- Su principal limitaci?n es la condici?n de estabilidad $r\le 1/2$.
- Para pasos de tiempo m?s grandes, conviene estudiar esquemas impl?citos (Backward Euler / Crank?Nicolson).